# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [3]:
# load and read data
df = pd.read_csv("AviationData.csv", encoding="latin1", low_memory= False)
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [4]:
# Check data type/info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Number.of.Engines,82805.0,1.146585,0.446510,0.0,1.0,1.0,1.0,8.0
Total.Fatal.Injuries,77488.0,0.647855,5.485960,0.0,0.0,0.0,0.0,349.0
Total.Serious.Injuries,76379.0,0.279881,1.544084,0.0,0.0,0.0,0.0,161.0
Total.Minor.Injuries,76956.0,0.357061,2.235625,0.0,0.0,0.0,0.0,380.0
Total.Uninjured,82977.0,5.325440,27.913634,0.0,0.0,1.0,2.0,699.0


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
# Columns relevant to the client's questions
relevant_cols = [
    'Event.Date',           
    'Make', 'Model',        
    'Aircraft.Category',    
    'Amateur.Built',        
    'Number.of.Engines',    
    'Engine.Type',          
    'Weather.Condition',    
    'Broad.phase.of.flight',
    'Injury.Severity',      
    'Aircraft.damage',      
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]
# Null counts sorted descending
print("***NULL COUNTS***")
print(df[relevant_cols].isna().sum().sort_values(ascending=False))

print("\n***DTYPES***")
print(df[relevant_cols].dtypes)



***NULL COUNTS***
Aircraft.Category         56602
Broad.phase.of.flight     27165
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Injury.Severity            1000
Amateur.Built               102
Model                        92
Make                         63
Event.Date                    0
dtype: int64

***DTYPES***
Event.Date                 object
Make                       object
Model                      object
Aircraft.Category          object
Amateur.Built              object
Number.of.Engines         float64
Engine.Type                object
Weather.Condition          object
Broad.phase.of.flight      object
Injury.Severity            object
Aircraft.damage            object
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64

In [7]:
# Inspect unique values in key categorical columns
for col in ["Aircraft.Category", "Amateur.Built", "Weather.Condition", "Engine.Type", "Aircraft.damage", "Injury.Severity"]:
    print("{=*30}")
    print(f" {col} (unique={df[col].nunique()})")
    print("{=*30}")
    print(df[col].value_counts(dropna=False).to_string())

{=*30}
 Aircraft.Category (unique=15)
{=*30}
Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
{=*30}
 Amateur.Built (unique=2)
{=*30}
Amateur.Built
No     80312
Yes     8475
NaN      102
{=*30}
 Weather.Condition (unique=4)
{=*30}
Weather.Condition
VMC    77303
IMC     5976
NaN     4492
UNK      856
Unk      262
{=*30}
 Engine.Type (unique=12)
{=*30}
Engine.Type
Reciprocating      69530
NaN                 7096
Turbo Shaft         3609
Turbo Prop          3391
Turbo Fan           2481
Unknown             2051
Turbo Jet            703
Geared Turbofan       12
Electric              10
LR       

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [8]:
#Cleaning and derived metrics for injury severity
# Convert injury columns to numeric
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Imputing missing njury counts as 0
df[injury_cols] = df[injury_cols].fillna(0)

# -----------------------------
# 3. Estimate total passengers onboard
# Assumption:
# Sum of all known outcomes = people onboard
# fatal + serious + minor + uninjured
# -----------------------------
df["Estimated.Passengers"] = (
    df["Total.Fatal.Injuries"]
    + df["Total.Serious.Injuries"]
    + df["Total.Minor.Injuries"]
    + df["Total.Uninjured"]
)

# Replacing zeros with NaN
df.loc[df["Estimated.Passengers"] == 0, "Estimated.Passengers"] = np.nan

# Fatal plus serious injuries count
df["Fatal.Serious.Count"] = (
    df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
)

# The Fraction of passengers who were killed or seriously injured
df["Fatal.Serious.Rate"] = (
    df["Fatal.Serious.Count"] / df["Estimated.Passengers"]
)
df["Injury.Risk.Level"] = pd.cut(
    df["Fatal.Serious.Rate"],
    bins=[-0.01, 0, 0.25, 0.50, 1.0],
    labels=["None", "Low", "Moderate", "High"]
)
# Inspecting results
df[
    [
        "Total.Fatal.Injuries",
        "Total.Serious.Injuries",
        "Estimated.Passengers",
        "Fatal.Serious.Count",
        "Fatal.Serious.Rate",
        "Injury.Risk.Level"
    ]
].head()

,Total.Fatal.Injuries,Total.Serious.Injuries,Estimated.Passengers,Fatal.Serious.Count,Fatal.Serious.Rate,Injury.Risk.Level
0,2.0,0.0,2.0,2.0,1.0,High
1,4.0,0.0,4.0,4.0,1.0,High
2,3.0,0.0,3.0,3.0,1.0,High
3,2.0,0.0,2.0,2.0,1.0,High
4,1.0,2.0,3.0,3.0,1.0,High


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [9]:
#Cleaning aircraft damage column(destruction Robustness)
print(df["Aircraft.damage"].value_counts(dropna=False))

df["Aircraft.damage"] = (
    df["Aircraft.damage"]
    .astype(str)
    .str.strip()
    .str.title()
)
# Convert placeholder strings back to NaN
df["Aircraft.damage"] = df["Aircraft.damage"].replace(
    ["Nan", "Unknown", "Unk", ""], np.nan
)
damage_map = {
    "Destroyed": "Destroyed",
    "Substantial": "Damaged",
    "Minor": "Damaged",
    "None": "No Damage"
}

df["Aircraft.Damage.Clean"] = df["Aircraft.damage"].map(damage_map)

# Anything that is unmatched becomes Unknown
df["Aircraft.Damage.Clean"] = df["Aircraft.Damage.Clean"].fillna("Unknown")

df["Aircraft.Destroyed"] = np.where(
    df["Aircraft.Damage.Clean"] == "Destroyed", 1, 0
)
#Robustness to destruction metric
df["Aircraft.Robustness"] = 1 - df["Aircraft.Destroyed"]

# Inspect
df[
    [
        "Aircraft.damage",
        "Aircraft.Damage.Clean",
        "Aircraft.Destroyed",
        "Aircraft.Robustness"
    ]
].head()

Aircraft.damage
Substantial    64148
Destroyed      18623
NaN             3194
Minor           2805
Unknown          119
Name: count, dtype: int64


,Aircraft.damage,Aircraft.Damage.Clean,Aircraft.Destroyed,Aircraft.Robustness
0,Destroyed,Destroyed,1,0
1,Destroyed,Destroyed,1,0
2,Destroyed,Destroyed,1,0
3,Destroyed,Destroyed,1,0
4,Destroyed,Destroyed,1,0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [10]:
# Triming whitespace
df["Make_clean"] = df["Make"].astype(str).str.strip()

df["Make_clean"] = df["Make_clean"].str.title()
# Checking for inconsistencies
corrections = {
    "Mcdonnell Douglas": "McDonnell Douglas",
    "Mc Donnell Douglas": "McDonnell Douglas",
    "De Havilland": "De Havilland",
    "Beechcraft": "Beech",
    "Bell Helicopter": "Bell",
    "Aerospatiale": "Aérospatiale"
}

df["Make_clean"] = df["Make_clean"].replace(corrections)

# Checking frequency before filtering
make_counts = df["Make_clean"].value_counts()

# 5. Applying the threshold (>= 50)
threshold = 50
valid_makes = make_counts[make_counts >= threshold].index

df_filtered = df[df["Make_clean"].isin(valid_makes)].copy()

print("Top Makes After Cleaning:")
print(df_filtered["Make_clean"].value_counts().head(10))

print("\nTotal unique makes retained:", df_filtered["Make_clean"].nunique())

Top Makes After Cleaning:
Make_clean
Cessna      27149
Piper       14870
Beech        5402
Boeing       2745
Bell         2729
Mooney       1334
Robinson     1230
Grumman      1172
Bellanca     1045
Hughes        932
Name: count, dtype: int64

Total unique makes retained: 99


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [11]:
# 1. Inspection
print("Initial info:")
print(df[["Make", "Model"]].info())

print("\nMissing values in Model:")
print(df["Model"].isna().sum())

# Drop rows where Model is missing
df = df.dropna(subset=["Model"]).copy()

print("\nAfter dropping NaNs:")
print(df["Model"].isna().sum())

# Cleaning Model column
df["Model_clean"] = (
    df["Model"]
    .astype(str)
    .str.strip()
    .str.upper()
)
# Counting by model
model_counts = df["Model_clean"].value_counts()
print("\nTop Models:")
print(model_counts.head(10))

# Counting by model and make
make_model_counts = (
    df.groupby(["Make_clean", "Model_clean"])
      .size()
      .sort_values(ascending=False)
)

print("\nTop Make-Model combinations:")
print(make_model_counts.head(10))

# Checking if models are unique across makes
model_make_variation = (
    df.groupby("Model_clean")["Make_clean"]
      .nunique()
      .sort_values(ascending=False)
)

print("\nModels appearing under multiple Makes:")
print(model_make_variation[model_make_variation > 1].head(10))

# Creating unique aircraft identifier(Make + Model)
df["Aircraft_Type"] = df["Make_clean"] + " " + df["Model_clean"]

print("\nTop Aircraft Types:")
print(df["Aircraft_Type"].value_counts().head(10))

print("\nUnique Aircraft Types:", df["Aircraft_Type"].nunique())

Initial info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Make    88826 non-null  object
 1   Model   88797 non-null  object
dtypes: object(2)
memory usage: 1.4+ MB
None

Missing values in Model:
92

After dropping NaNs:
0

Top Models:
Model_clean
152          2367
172          1756
172N         1164
PA-28-140     932
150           829
172M          798
172P          689
182           659
180           622
150M          585
Name: count, dtype: int64

Top Make-Model combinations:
Make_clean  Model_clean
Cessna      152            2366
            172            1753
            172N           1163
Piper       PA-28-140       932
Cessna      150             829
            172M            798
            172P            689
            182             659
            180             621
            150M            585
dtype: int64

Models appearing 

Observation: The Model names are not unowue across manufactures hence the creation of unique identifier for a given plane type

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [12]:
def clean_categorical(series):
    return (
        series.astype(str)
        .str.strip()
        .str.title()
        .replace(["", "Nan", "None", "Unknown", "Unk"], np.nan)
    )

# 1. Checking for Engine Type
df["Engine.Type_clean"] = clean_categorical(df["Engine.Type"])

engine_map = {
    "Reciprocating": "Reciprocating",
    "Turbo Prop": "Turboprop",
    "Turbo-Prop": "Turboprop",
    "Turbo Jet": "Turbojet",
    "Turbo-Fan": "Turbofan"
}
df["Engine.Type_clean"] = df["Engine.Type_clean"].replace(engine_map)

print("\nEngine Type Counts:")
print(df["Engine.Type_clean"].value_counts(dropna=False))

# 2. Checking Weather.Condition
df["Weather.Condition_clean"] = (
    df["Weather.Condition"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace(["", "UNKNOWN", "UNK"], np.nan)
)

# Checking to keep only expected categories
valid_weather = ["VMC", "IMC"]
df.loc[~df["Weather.Condition_clean"].isin(valid_weather), "Weather.Condition_clean"] = np.nan

print("\nWeather Condition Counts:")
print(df["Weather.Condition_clean"].value_counts(dropna=False))


# 3. Number.of.Engines
df["Number.of.Engines_clean"] = pd.to_numeric(
    df["Number.of.Engines"], errors="coerce"
)

# Replacing unrealistic values
df.loc[
    (df["Number.of.Engines_clean"] <= 0) |
    (df["Number.of.Engines_clean"] > 8),  # typical aircraft rarely exceed this
    "Number.of.Engines_clean"
] = np.nan

print("\nNumber of Engines Stats:")
print(df["Number.of.Engines_clean"].describe())


# 4. Purpose.of.flight
df["Purpose.of.flight_clean"] = clean_categorical(df["Purpose.of.flight"])

# Group rare categories
threshold = 50
purpose_counts = df["Purpose.of.flight_clean"].value_counts()
rare_purposes = purpose_counts[purpose_counts < threshold].index

df["Purpose.of.flight_grouped"] = df["Purpose.of.flight_clean"].replace(
    rare_purposes, "Other"
)

print("\nPurpose of Flight Counts:")
print(df["Purpose.of.flight_grouped"].value_counts())


# 5. Broad.phase.of.flight
df["Broad.phase.of.flight_clean"] = clean_categorical(df["Broad.phase.of.flight"])

# common variations
phase_map = {
    "Take-Off": "Takeoff",
    "Take Off": "Takeoff",
    "Landing Roll": "Landing",
    "Approach": "Approach",
    "Climb": "Climb",
    "Cruise": "Cruise",
    "Descent": "Descent"
}
df["Broad.phase.of.flight_clean"] = df["Broad.phase.of.flight_clean"].replace(phase_map)

# Group rare categories
phase_counts = df["Broad.phase.of.flight_clean"].value_counts()
rare_phases = phase_counts[phase_counts < threshold].index

df["Broad.phase.of.flight_grouped"] = df["Broad.phase.of.flight_clean"].replace(
    rare_phases, "Other"
)

print("\nPhase of Flight Counts:")
print(df["Broad.phase.of.flight_grouped"].value_counts())


Engine Type Counts:
Engine.Type_clean
Reciprocating      69506
NaN                 9086
Turbo Shaft         3609
Turboprop           3390
Turbo Fan           2478
Turbojet             703
Geared Turbofan       12
Electric              10
Lr                     2
Hybrid Rocket          1
Name: count, dtype: int64

Weather Condition Counts:
Weather.Condition_clean
VMC    77264
IMC     5974
NaN     5559
Name: count, dtype: int64

Number of Engines Stats:
count    81540.000000
mean         1.163809
std          0.427001
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          8.000000
Name: Number.of.Engines_clean, dtype: float64

Purpose of Flight Counts:
Purpose.of.flight_grouped
Personal                     49423
Instructional                10599
Aerial Application            4710
Business                      4016
Positioning                   1645
Other Work Use                1264
Ferry                          812
Aerial Observation     

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [13]:
# 1. Inspect non-null counts
non_null_counts = df.notna().sum().sort_values(ascending=False)

print("Non-null counts per column:")
print(non_null_counts)

# 2. Applying threshold
threshold = 20000

# Columns to KEEP
cols_to_keep = non_null_counts[non_null_counts > threshold].index

# Columns to DROP
cols_to_drop = non_null_counts[non_null_counts <= threshold].index

print("\nColumns to KEEP (>20,000 non-nulls):")
print(list(cols_to_keep))

print("\nColumns to DROP (≤20,000 non-nulls):")
print(list(cols_to_drop))


# 3. Filtered dataframe
df_reduced = df[cols_to_keep].copy()

print("\nShape before:", df.shape)
print("Shape after:", df_reduced.shape)


Non-null counts per column:
Event.Id                         88797
Model                            88797
Aircraft_Type                    88797
Model_clean                      88797
Make_clean                       88797
Aircraft.Robustness              88797
Aircraft.Destroyed               88797
Aircraft.Damage.Clean            88797
Fatal.Serious.Count              88797
Total.Uninjured                  88797
Total.Minor.Injuries             88797
Investigation.Type               88797
Total.Fatal.Injuries             88797
Total.Serious.Injuries           88797
Accident.Number                  88797
Event.Date                       88797
Make                             88777
Location                         88745
Amateur.Built                    88697
Country                          88572
Injury.Severity                  87817
Injury.Risk.Level                87516
Fatal.Serious.Rate               87516
Estimated.Passengers             87516
Registration.Number              874

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [14]:
df.to_csv("AviationData_Cleaned_Final.csv", index=False)